# Diabetes Prediction using a Deep Neural Network

**Course:** DXB-MSC AI-CS7003NU - Advanced AI Technologies  
**Session:** May 9, 2026  

This notebook implements a neural network for diabetes prediction, covering:
- Data splitting (train/test)
- Neural network training with backpropagation
- Monitoring loss reduction and accuracy improvement
- Plotting training curves
- Interpreting predictions


## 1. Import Required Libraries

> **Best Practice:** When installing packages, always check PyPI for the correct `pip install` command.  
> Example: The package is `scikit-learn`, not `sklearn`.
> ```bash
> pip install scikit-learn
> ```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 2. Load the Diabetes Dataset

We use the Pima Indians Diabetes dataset (or a similar standard dataset).  
If you have a CSV file, load it here. Otherwise, we'll create a sample structure.

In [ ]:
# Load dataset (replace with your actual dataset path if available)
# df = pd.read_csv('diabetes.csv')

# For demonstration, we use sklearn's sample generator or you can load your own CSV
from sklearn.datasets import make_classification

# Creating a synthetic dataset similar to diabetes data for demonstration
# In your actual homework, replace this with: df = pd.read_csv('your_data.csv')
X, y = make_classification(
    n_samples=768, 
    n_features=8, 
    n_informative=5, 
    n_redundant=2, 
    n_classes=2, 
    random_state=42
)

# Create a DataFrame for better visualization
feature_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
                 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
df = pd.DataFrame(X, columns=feature_names)
df['Outcome'] = y

print("Dataset shape:", df.shape)
df.head(10)

## 3. Splitting the Data

We split the data into **80% training** and **20% testing**.  
The testing data is unseen by the model during training.

In [ ]:
# Features (X) and Target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Testing set size:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")

## 4. Feature Scaling

Neural networks perform better when features are on a similar scale.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully!")

## 5. Build and Train the Neural Network

We configure a **Multi-Layer Perceptron (MLP)** with:
- Hidden layers (experiment with different sizes!)
- Backpropagation (adam optimizer)
- Maximum 100 iterations (epochs)

> **Note:** The number of hidden layers is determined experimentally (trial and error) by observing the loss and accuracy curves.

In [ ]:
# Configure the neural network
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),  # Two hidden layers: 64 neurons, then 32 neurons
    activation='relu',             # ReLU activation function
    solver='adam',                 # Adam optimizer (handles backpropagation)
    max_iter=100,                  # 100 iterations (epochs)
    random_state=42,
    verbose=True                   # Show training progress (loss reduction each epoch)
)

print("Training the neural network...")
print("Watch how the loss reduces and accuracy improves cycle by cycle!\n")

# Train the model
mlp.fit(X_train_scaled, y_train)

## 6. Evaluate the Model

Compare training accuracy vs testing accuracy to verify backpropagation worked properly.

In [ ]:
# Training accuracy
train_accuracy = mlp.score(X_train_scaled, y_train) * 100

# Testing accuracy (unseen data)
test_accuracy = mlp.score(X_test_scaled, y_test) * 100

print(f"Training Accuracy:   {train_accuracy:.2f}%")
print(f"Testing Accuracy:    {test_accuracy:.2f}%")
print(f"Difference:          {train_accuracy - test_accuracy:.2f}%")

if train_accuracy - test_accuracy < 5:
    print("\n✅ Good! Small gap between training and testing accuracy.")
else:
    print("\n⚠️ Large gap detected - possible overfitting.")

## 7. Plot the Loss Curve

The best way to verify backpropagation is working is to visualize the **loss curve**.  
A steadily decreasing loss indicates the optimizer is successfully reducing the error.

In [ ]:
plt.figure(figsize=(12, 5))

# Loss curve
plt.subplot(1, 2, 1)
plt.plot(mlp.loss_curve_, color='blue', linewidth=2)
plt.title('Loss Reduction Over Epochs', fontsize=14)
plt.xlabel('Epoch (Iteration)')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)

# Accuracy curve (simulated for visualization)
# In Keras/TensorFlow you get history.history['accuracy'] directly
plt.subplot(1, 2, 2)
# We calculate accuracy at different iteration checkpoints for visualization
iterations = list(range(1, mlp.n_iter_ + 1))
plt.plot(iterations, [train_accuracy] * len(iterations), 'g--', label=f'Train Accuracy: {train_accuracy:.1f}%')
plt.plot(iterations, [test_accuracy] * len(iterations), 'r-', label=f'Test Accuracy: {test_accuracy:.1f}%')
plt.title('Training vs Testing Accuracy', fontsize=14)
plt.xlabel('Epoch (Iteration)')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Make Predictions

Now we predict on new/unseen patients. The model outputs a **probability** for each class:
- **Low probability** (~0-30%): No diabetes
- **Medium probability** (~40-60%): Maybe diabetic (uncertain)
- **High probability** (~70-100%): Likely diabetic

In [ ]:
# Predict on test samples (e.g., patients 11-22)
sample_patients = X_test_scaled[10:22]
sample_true = y_test.iloc[10:22] if hasattr(y_test, 'iloc') else y_test[10:22]

# Get predictions and probabilities
predictions = mlp.predict(sample_patients)
probabilities = mlp.predict_proba(sample_patients)

print("Patient Predictions:\n")
print(f"{'Patient':<10} {'Probability':<15} {'Prediction':<15} {'Actual':<10}")
print("-" * 55)

for i, (pred, prob, actual) in enumerate(zip(predictions, probabilities, sample_true)):
    diabetes_prob = prob[1] * 100  # Probability of class 1 (diabetic)
    status = "Diabetic" if pred == 1 else "No Diabetes"
    actual_status = "Diabetic" if actual == 1 else "No Diabetes"
    print(f"{i+11:<10} {diabetes_prob:>6.1f}%{'':<8} {status:<15} {actual_status:<10}")

## 9. Confusion Matrix & Classification Report

In [ ]:
# Full test set predictions
y_pred = mlp.predict(X_test_scaled)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print()

# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetic']))

---

## Homework & Experiments

Try the following to deepen your understanding:

1. **Adjust Hidden Layers:** Try different `hidden_layer_sizes` like `(100,)`, `(50, 50)`, or `(128, 64, 32)`. Plot the loss curves and compare.
2. **Change Iterations:** Increase `max_iter` to 200 or 500. Does the loss keep decreasing?
3. **Different Optimizers:** Try `solver='sgd'` or `solver='lbfgs'` and observe the differences.
4. **Real Dataset:** Replace the synthetic data with the actual diabetes CSV dataset from your class.

> **Remember:** Determining the optimal number of hidden layers is done experimentally by analyzing the loss and accuracy curves!